In [115]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from utils import *

In [145]:
results = {"CV": np.array([]),
           "ACC": np.array([]),
           "PRE": np.array([]),
           "REC": np.array([]),
           "F1": np.array([])}

In [147]:
for i in range(50):

    comments, sentiments = load_data("dataset/reviews.csv")

    # create dictionary mapping from emoji to sentiment
    emoji_dict = create_emoji_dict(comments)
    for i in range(len(comments)):
        comments[i] = normalize_emoji(comments[i], emoji_dict)
        comments[i] = clean_text(comments[i])
        comments[i] = remove_stopwords(comments[i])

    # remove null comments
    mask = comments != ""
    comments = comments[mask]
    sentiments = sentiments[mask]

    permutations = np.random.permutation(len(comments))
    comments = comments[permutations]
    sentiments = sentiments[permutations]
    X_train, X_test, Y_train, Y_test = train_test_split(comments, sentiments, test_size=0.02)
    vectorizer = TfidfVectorizer(ngram_range=(1, 3))

    X_train_vector = vectorizer.fit_transform(X_train)
    X_test_vector = vectorizer.transform(X_test)
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    model = LogisticRegression(class_weight="balanced", max_iter=45000, solver="lbfgs")

    model.fit(X_train_vector, Y_train)


    score = cross_val_score(model, X_train_vector, Y_train, cv=kf, scoring="accuracy")
    results["CV"] = np.append(results["CV"], score.mean())
    # print(f"{'Cross-validation score:':<35}{score.mean():.4f}")



    preds = model.predict(X_test_vector)
    results["ACC"] = np.append(results["ACC"], accuracy_score(Y_test, preds))
    results["F1"] = np.append(results["F1"], f1_score(Y_test, preds, average="weighted"))
    results["REC"] = np.append(results["REC"], recall_score(Y_test, preds, average="weighted"))
    results["PRE"] = np.append(results["PRE"], precision_score(Y_test, preds, average="weighted"))

    # print(f"{'Accuracy:':<35}{acc:.4f}")
    # print(f"{'Precision:':<35}{precision:.4f}")
    # print(f"{'Recall:':<35}{recall:.4f}")
    # print(f"{'F1-score:':<35}{f1:.4f}")


In [150]:
for key in results.keys():
    print(key)
    print(f'max: {results[key].max()}')
    print(f'min: {results[key].min()}')
    print(f'average: {results[key].mean()}')

CV
max: 0.7453125
min: 0.7353515625
average: 0.73973046875
ACC
max: 0.819047619047619
min: 0.6666666666666666
average: 0.7380952380952381
PRE
max: 0.8284487914981661
min: 0.6997125165855815
average: 0.7578414173081193
REC
max: 0.819047619047619
min: 0.6666666666666666
average: 0.7380952380952381
F1
max: 0.819241431351523
min: 0.6790815503144271
average: 0.7415470752660355


In [83]:
# from sklearn.linear_model import SGDClassifier
# from sklearn.metrics import log_loss
# # Define SGD-based Logistic Regression
# sgd = SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.001, max_iter=1000)


# # Train
# sgd.fit(X_train_vector, Y_train)
# # Predict
# preds = sgd.predict(X_test_vector)

# # Evaluate
# acc = accuracy_score(Y_test, preds)
# f1 = f1_score(Y_test, preds, average="weighted")
# recall = recall_score(Y_test, preds, average="weighted")
# precision = precision_score(Y_test, preds, average="weighted")
# cross_validation_score = cross_val_score(model, X_train_vector, Y_train, cv=kf, scoring="accuracy")

# print("Accuracy:", acc)
# print("Precision:", precision)
# print("Recall:", recall)
# print("F1-score:", f1)
# print("SC-Score: ", cross_validation_score)
